In [ ]:
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

# set directories
WD_junxi = Path('PATH_TO_DATA')
WD = WD_junxi
data_dir = Path(WD/'EntTemplates/Analysis/python_Patent/data/')
output_dir = Path(WD/'EntTemplates/Analysis/python_Patent/output/')
# Ensure prediction output directory exists
pred_output_dir = output_dir / "patent_predictions_100k_china"
pred_output_dir.mkdir(parents=True, exist_ok=True)
patent_model_dir = Path("PATH_TO_PATENT_MODEL_FOLDERS")

BERT_output_dir = Path(WD / 'EntTemplates/Analysis/python_BERT/data_patent/positive/')
import os
os.environ["WANDB_DISABLED"] = "true" 

In [2]:
g_patent = pd.read_csv(data_dir / 'g_patent.tsv', sep='\t', low_memory=False) 
g_patent = g_patent[g_patent['patent_type'] == 'utility']
g_assignee = pd.read_csv(data_dir / 'g_assignee_disambiguated.tsv', sep='\t', low_memory=False)
g_assignee = g_assignee[g_assignee['assignee_sequence'] == 0]
g_assignee = g_assignee[['patent_id', 'assignee_id', 'disambig_assignee_organization', 'location_id']]
g_patent = g_patent.merge(g_assignee, on='patent_id', how='inner')
del g_assignee
g_location = pd.read_csv(data_dir / 'g_location_disambiguated.tsv', sep='\t', low_memory=False)
g_patent = g_patent.merge(g_location[['location_id', 'disambig_country']], on='location_id', how='left')
pb_assignee = pd.read_excel(data_dir / 'matched_result.xlsx')
pb_marketmap = pd.read_csv(data_dir / 'predicted_positive_v2.csv')
pb_marketmap.sort_values(by=['companyid'], inplace=True)
pb_marketmap['count_sector'] = pb_marketmap.groupby('companyid')['companyid'].transform('count')
pb_marketmap['unique_segment'] = pb_marketmap['marketmap'] + pb_marketmap['segment']
pb_marketmap['count_segment'] = pb_marketmap.groupby('companyid')['unique_segment'].transform('nunique')
pb_marketmap['count_marketmap'] = pb_marketmap.groupby('companyid')['marketmap'].transform('nunique')
pb_marketmap = pb_marketmap[pb_marketmap['count_marketmap'] == 1]
pb_marketmap = pd.merge(pb_marketmap, pb_assignee[['companyid', 'assignee_id']], on='companyid', how='inner')
df_training = pd.merge(g_patent, pb_marketmap, on='assignee_id', how='inner')
df_predicting = g_patent[~g_patent['patent_id'].isin(df_training['patent_id'])]
print(f"Total patents for predicting: {df_predicting.shape[0]}")
#####################################
# keep if disambig_country is not US
#####################################
df_predicting = df_predicting[df_predicting['disambig_country'] != 'US']
print(f"Patents for predicting after removing US: {df_predicting.shape[0]}")
# turn patent_date into datetime
df_predicting['patent_date'] = pd.to_datetime(df_predicting['patent_date'], errors='coerce')
# keep if patent between 2000 and 2019
df_predicting = df_predicting[(df_predicting['patent_date'].dt.year >= 2000) & (df_predicting['patent_date'].dt.year <= 2019)]
print(f"Patents for predicting after date filter: {df_predicting.shape[0]}" )
# keep if abstract is not null
df_predicting = df_predicting[~df_predicting['patent_abstract'].isnull()]
print(f"Patents for predicting after abstract not null filter: {df_predicting.shape[0]}" )
# keep if disambig_country not null
df_predicting = df_predicting[~df_predicting['disambig_country'].isnull()]
# keep if abstract as str after strip is not empty
df_predicting = df_predicting[df_predicting['patent_abstract'].str.strip() != '']
print(f"Patents for predicting after disambig_country not null filter: {df_predicting.shape[0]}" )
# sample 100k patents
df_predicting = df_predicting.sample(n=100000, random_state=42)
# save predicting dataframe
df_predicting.to_csv(pred_output_dir / 'df_predicting_100k_china.tsv', sep='\t', index=False)

Total patents for predicting: 6451102
Patents for predicting after removing US: 3241472
Patents for predicting after date filter: 1991128
Patents for predicting after abstract not null filter: 1990818
Patents for predicting after disambig_country not null filter: 1972560


In [4]:
df = pd.read_csv(pred_output_dir / 'df_predicting_100k_china.tsv', sep='\t', low_memory=False)
df_pred = df[["patent_id", "patent_abstract"]].copy()
df_pred = df_pred.rename(columns={"patent_id": "appid", "patent_abstract": "abstract"})
df_pred["appid"] = df_pred["appid"].astype(str)
df_pred = df_pred[df_pred["abstract"].str.strip() != ""].reset_index(drop=True)

In [6]:
# Discover all subsegment model folders
model_base = patent_model_dir
model_folders = [p for p in model_base.glob("SUBSEGMENT-*") if (p / "best_model" / "model.safetensors").exists()]
# sort by subsegment name for consistent processing order
model_folders = sorted(model_folders, key=lambda p: p.name)
print(f"Discovered {len(model_folders)} subsegment model folders.")
results_collection = []

# Use the original base tokenizer (not saved in best_model)
tokenizer_base = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_base)

hf_ds = Dataset.from_pandas(df_pred, preserve_index=False)

def safe_tokenize_function(examples):
    return tokenizer(examples["abstract"], padding="max_length", truncation=True, max_length=512)

predict_dataset = hf_ds.map(safe_tokenize_function, batched=True, remove_columns=hf_ds.column_names)

for model_root in model_folders:
    subsegment = model_root.name.replace("SUBSEGMENT-", "")
    out_path_to_check = pred_output_dir / f"{subsegment}_patent_positive_bert.csv"

    # Skip if results already exist
    if out_path_to_check.exists():
        print(f"Skipping {subsegment}: output already exists at {out_path_to_check.name}")
        continue

    print(f"Processing subsegment: {subsegment}. Total number of sectors: {len(model_folders)}")
    model_path = model_root / "best_model"  # weights live here

    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    trainer = Trainer(model=model)

    predictions = trainer.predict(predict_dataset)
    logits = predictions.predictions
    y_neg = logits[:, 0]
    y_pos = logits[:, 1]
    y_pred = np.argmax(logits, axis=-1)

    df_results = pd.DataFrame({
        "appid": df_pred["appid"],
        "abstract": df_pred["abstract"],
        "pred_pos": y_pos,
        "pred_neg": y_neg,
        "positive": y_pred,
        "sector": subsegment,
    })
    out_path = pred_output_dir / f"{subsegment}_patent_positive_bert.csv"
    df_results.to_csv(out_path, index=False)
    results_collection.append((subsegment, df_results.head()))

Discovered 217 subsegment model folders.


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Skipping AIMLAIMLSemiconductorsEdgeAISoftware: output already exists at AIMLAIMLSemiconductorsEdgeAISoftware_patent_positive_bert.csv
Skipping AIMLAIMLSemiconductorsIntelligentSensorsDevices: output already exists at AIMLAIMLSemiconductorsIntelligentSensorsDevices_patent_positive_bert.csv
Skipping AIMLAIMLSemiconductorsProcessorDesign: output already exists at AIMLAIMLSemiconductorsProcessorDesign_patent_positive_bert.csv
Skipping AIMLAutonomousMachinesIntelligentRobotics: output already exists at AIMLAutonomousMachinesIntelligentRobotics_patent_positive_bert.csv
Skipping AIMLHorizontalPlatformsAIAutomationPlatforms: output already exists at AIMLHorizontalPlatformsAIAutomationPlatforms_patent_positive_bert.csv
Skipping AIMLHorizontalPlatformsAICore: output already exists at AIMLHorizontalPlatformsAICore_patent_positive_bert.csv
Skipping AIMLHorizontalPlatformsComputerVision: output already exists at AIMLHorizontalPlatformsComputerVision_patent_positive_bert.csv
Skipping AIMLHorizontalP